In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from scipy.optimize import minimize
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import lightgbm as lgb
from catboost import CatBoostClassifier
import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)
print('All imports OK')

In [ ]:

# SECTION 1: DATA LOADING & UNDERSTANDING

import pandas as pd
import numpy as np

# Load the datasets
print("Loading data...")
train = pd.read_csv("d:/datasets/train.csv")
test = pd.read_csv("d:/datasets/test.csv")

# Check the shape (rows x columns)
print(f"\nTrain data: {train.shape[0]} rows, {train.shape[1]} columns")
print(f"Test data:  {test.shape[0]} rows, {test.shape[1]} columns")

# Look at first few rows
print("\n--- First 5 rows of training data ---")
print(train.head())

# Check column names
print("\n--- Column names ---")
print(train.columns.tolist())

# Check data types
print("\n--- Data types ---")
print(train.dtypes)

# Basic statistics
print("\n--- Basic statistics ---")
print(train.describe())

# Check target variable (what we're predicting)
print("\n--- Target variable distribution ---")
print(train['Churn'].value_counts())
print(f"\nChurn rate: {(train['Churn'] == 'Yes').sum() / len(train) * 100:.2f}%")


In [ ]:
# ============================================================
# SECTION 2: DATA CLEANING
# ============================================================

print("=" * 60)
print("SECTION 2: DATA CLEANING")
print("=" * 60)

# Step 1: Check for missing values
print("\n--- Checking for missing values ---")
print("\nTraining data missing values:")
missing_train = train.isnull().sum()
print(missing_train[missing_train > 0])  # Only show columns with missing data

print("\nTest data missing values:")
missing_test = test.isnull().sum()
print(missing_test[missing_test > 0])

# Step 2: Fix TotalCharges column (it has spaces that cause problems)
print("\n--- Fixing TotalCharges column ---")

# Convert TotalCharges to numeric (spaces will become NaN)
train['TotalCharges'] = pd.to_numeric(train['TotalCharges'], errors='coerce')
test['TotalCharges'] = pd.to_numeric(test['TotalCharges'], errors='coerce')

# Check how many NaN we created
print(f"NaN values in train TotalCharges: {train['TotalCharges'].isnull().sum()}")
print(f"NaN values in test TotalCharges: {test['TotalCharges'].isnull().sum()}")

# Fill missing TotalCharges with median (middle value)
median_charges = train['TotalCharges'].median()
print(f"\nFilling missing values with median: {median_charges:.2f}")

train['TotalCharges'].fillna(median_charges, inplace=True)
test['TotalCharges'].fillna(median_charges, inplace=True)

# Step 3: Verify - no missing values left
print("\n--- After cleaning ---")
print(f"Missing values in train: {train.isnull().sum().sum()}")
print(f"Missing values in test: {test.isnull().sum().sum()}")

# Step 4: Save IDs and target separately
print("\n--- Separating IDs and target variable ---")
train_ids = train['id']
test_ids = test['id']

# Convert target from Yes/No to 1/0
train['Churn'] = train['Churn'].map({'Yes': 1, 'No': 0})
y = train['Churn']  # This is what we're predicting

print(f"Target variable (y) shape: {y.shape}")
print(f"Positive class (Churn=1): {y.sum()} customers ({y.mean()*100:.2f}%)")

print("\n✅ Data cleaning complete!")

In [ ]:
train.head(2)

In [ ]:

# ============================================================
# SECTION 3: EXPLORATORY DATA ANALYSIS (EDA)
# ============================================================

# import matplotlib.pyplot as plt
# import seaborn as sns

# Set style for prettier charts
# sns.set_style("whitegrid")
# plt.rcParams['figure.figsize'] = (12, 6)

print("=" * 60)
print("SECTION 3: EXPLORATORY DATA ANALYSIS")
print("=" * 60)

# ------------------------------------------------
# 3.1: Target Distribution
# ------------------------------------------------
print("\n--- 3.1: Target Variable Distribution ---")
churn_counts = train['Churn'].value_counts()
# plt.figure(figsize=(8, 5))
# plt.bar(['No Churn (0)', 'Churn (1)'], churn_counts.values, color=['green', 'red'])
# plt.title('Customer Churn Distribution', fontsize=14, fontweight='bold')
# plt.ylabel('Number of Customers')
# plt.xlabel('Churn Status')
# for i, v in enumerate(churn_counts.values):
#     plt.text(i, v + 5000, str(v), ha='center', fontweight='bold')
# plt.tight_layout()
# plt.show()

# ------------------------------------------------
# 3.2: Numerical Features Analysis
# ------------------------------------------------
print("\n--- 3.2: Numerical Features Analysis ---")

# Select numerical columns
numerical_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']

# fig, axes = plt.subplots(1, 3, figsize=(15, 4))
# for idx, col in enumerate(numerical_cols):
#     train.boxplot(column=col, by='Churn', ax=axes[idx])
#     axes[idx].set_title(f'{col} vs Churn')
#     axes[idx].set_xlabel('Churn (0=No, 1=Yes)')
#     axes[idx].set_ylabel(col)
# plt.suptitle('Numerical Features by Churn Status', fontsize=14, fontweight='bold')
# plt.tight_layout()
# plt.show()

# Statistics by churn
print("\nAverage values by Churn status:")
print(train.groupby('Churn')[numerical_cols].mean())

# ------------------------------------------------
# 3.3: Categorical Features Analysis
# ------------------------------------------------
print("\n--- 3.3: Categorical Features Analysis ---")

# Important categorical columns
categorical_cols = ['Contract', 'InternetService', 'PaymentMethod', 'gender']

# fig, axes = plt.subplots(2, 2, figsize=(14, 10))
# axes = axes.ravel()
# for idx, col in enumerate(categorical_cols):
#     churn_data = train.groupby([col, 'Churn']).size().unstack()
#     churn_data.plot(kind='bar', ax=axes[idx], color=['green', 'red'])
#     axes[idx].set_title(f'{col} vs Churn', fontweight='bold')
#     axes[idx].set_xlabel(col)
#     axes[idx].set_ylabel('Count')
#     axes[idx].legend(['No Churn', 'Churn'])
#     axes[idx].tick_params(axis='x', rotation=45)
# plt.tight_layout()
# plt.show()

# ------------------------------------------------
# 3.4: Churn Rate by Categories
# ------------------------------------------------
print("\n--- 3.4: Churn Rate by Category ---")

for col in categorical_cols:
    churn_rate = train.groupby(col)['Churn'].mean().sort_values(ascending=False)
    print(f"\n{col}:")
    for category, rate in churn_rate.items():
        print(f"  {category:30s}: {rate*100:5.2f}%")

# ------------------------------------------------
# 3.5: Correlation Matrix (Numerical Features)
# ------------------------------------------------
print("\n--- 3.5: Correlation Analysis ---")

corr_cols = ['Churn', 'tenure', 'MonthlyCharges', 'TotalCharges', 'SeniorCitizen']
correlation = train[corr_cols].corr()

# plt.figure(figsize=(8, 6))
# sns.heatmap(correlation, annot=True, cmap='coolwarm', center=0,
#             fmt='.3f', square=True, linewidths=1)
# plt.title('Correlation Matrix', fontsize=14, fontweight='bold')
# plt.tight_layout()
# plt.show()

print("\nCorrelation with Churn:")
print(correlation['Churn'].sort_values(ascending=False))

print("\n✅ EDA complete!")


In [ ]:

from sklearn.preprocessing import LabelEncoder

print("=" * 60)
print("SECTION 4: PREPROCESSING")
print("=" * 60)

# ------------------------------------------------
# 4.1: Remove ID columns (not useful for prediction)
# ------------------------------------------------
print("\n--- 4.1: Removing ID columns ---")
train_clean = train.drop(['id'], axis=1)
test_clean = test.drop(['id'], axis=1)

print(f"Train shape after removing ID: {train_clean.shape}")
print(f"Test shape after removing ID: {test_clean.shape}")

In [ ]:
print("\n--- 4.2: Separating features and target ---")

# Remove target from train to get features only
X_train_full = train_clean.drop(['Churn'], axis=1)
y_train_full = train_clean['Churn']

X_test_full = test_clean.copy()

print(f"Features (X_train_full): {X_train_full.shape}")
print(f"Target (y_train_full): {y_train_full.shape}")
print(f"Test features (X_test_full): {X_test_full.shape}")

# ------------------------------------------------
# 4.3: Identify column types
# ------------------------------------------------
print("\n--- 4.3: Identifying column types ---")

# Numerical columns
numerical_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']

# Binary columns (Yes/No or Male/Female)
binary_cols = ['gender', 'Partner', 'Dependents', 'PhoneService', 
               'PaperlessBilling']

# Multi-category columns
categorical_cols = ['MultipleLines', 'InternetService', 'OnlineSecurity',
                   'OnlineBackup', 'DeviceProtection', 'TechSupport',
                   'StreamingTV', 'StreamingMovies', 'Contract', 
                   'PaymentMethod']

print(f"Numerical columns ({len(numerical_cols)}): {numerical_cols}")
print(f"Binary columns ({len(binary_cols)}): {binary_cols}")
print(f"Categorical columns ({len(categorical_cols)}): {categorical_cols}")


In [ ]:
print("\n--- 4.4: Encoding binary columns ---")

# Combine train and test for consistent encoding
X_train_full['dataset'] = 'train'
X_test_full['dataset'] = 'test'
X_combined = pd.concat([X_train_full, X_test_full], axis=0, ignore_index=True)

# Encode binary columns
le = LabelEncoder()
for col in binary_cols:
    X_combined[col] = le.fit_transform(X_combined[col])
    print(f"  {col}: {X_combined[col].unique()[:5]}")  # Show first 5 unique values

# ------------------------------------------------
# 4.5: One-Hot Encode Categorical Columns
# ------------------------------------------------
print("\n--- 4.5: One-hot encoding categorical columns ---")
print("Before encoding:", X_combined.shape)

# One-hot encode (creates new columns for each category)
X_combined = pd.get_dummies(X_combined, columns=categorical_cols, 
                            drop_first=True, dtype=int)

print("After encoding:", X_combined.shape)
print(f"New columns created: {X_combined.shape[1] - len(numerical_cols) - len(binary_cols) - 1}")


In [ ]:

# ------------------------------------------------
print("\n--- 4.6: Splitting back to train/test ---")

X_train_processed = X_combined[X_combined['dataset'] == 'train'].drop(['dataset'], axis=1)
X_test_processed = X_combined[X_combined['dataset'] == 'test'].drop(['dataset'], axis=1)

print(f"X_train_processed: {X_train_processed.shape}")
print(f"X_test_processed: {X_test_processed.shape}")

# ------------------------------------------------
# 4.7: Final Check
# ------------------------------------------------
print("\n--- 4.7: Final verification ---")

# Check for missing values
print(f"Missing values in X_train: {X_train_processed.isnull().sum().sum()}")
print(f"Missing values in X_test: {X_test_processed.isnull().sum().sum()}")

# Check columns match
print(f"Train and test columns match: {list(X_train_processed.columns) == list(X_test_processed.columns)}")

# Show final column names
print(f"\nFinal features ({len(X_train_processed.columns)}):")
print(X_train_processed.columns.tolist())

print("\n✅ Preprocessing complete!")

In [ ]:
# ============================================================
# SECTION 5: TRAIN/TEST SPLIT
# ============================================================

from sklearn.model_selection import train_test_split

print("=" * 60)
print("SECTION 5: TRAIN/TEST SPLIT")
print("=" * 60)

# ------------------------------------------------
# 5.1: Split the data (80% train, 20% validation)
# ------------------------------------------------
print("\n--- 5.1: Splitting data ---")

X_train, X_val, y_train, y_val = train_test_split(
    X_train_processed,      # Features
    y_train_full,           # Target
    test_size=0.2,          # 20% for validation
    random_state=42,        # For reproducibility
    stratify=y_train_full   # Keep same churn ratio in both sets
)

print(f"Training set:   {X_train.shape[0]} samples ({X_train.shape[0]/len(X_train_processed)*100:.1f}%)")
print(f"Validation set: {X_val.shape[0]} samples ({X_val.shape[0]/len(X_train_processed)*100:.1f}%)")

# ------------------------------------------------
# 5.2: Verify class distribution
# ------------------------------------------------
print("\n--- 5.2: Verifying class distribution ---")

train_churn_rate = y_train.mean() * 100
val_churn_rate = y_val.mean() * 100
original_churn_rate = y_train_full.mean() * 100

print(f"Original churn rate:    {original_churn_rate:.2f}%")
print(f"Training churn rate:    {train_churn_rate:.2f}%")
print(f"Validation churn rate:  {val_churn_rate:.2f}%")

# Check if they're similar (should be within 1%)
if abs(train_churn_rate - val_churn_rate) < 1:
    print("✅ Class distribution is balanced!")
else:
    print("⚠️ Warning: Class distribution might be unbalanced")

# ------------------------------------------------
# 5.3: Summary
# ------------------------------------------------
print("\n--- 5.3: Final dataset summary ---")
print(f"\nTraining data:")
print(f"  X_train: {X_train.shape} (features)")
print(f"  y_train: {y_train.shape} (target)")
print(f"  Churn=1: {y_train.sum()} ({y_train.mean()*100:.2f}%)")
print(f"  Churn=0: {len(y_train) - y_train.sum()} ({(1-y_train.mean())*100:.2f}%)")

print(f"\nValidation data:")
print(f"  X_val: {X_val.shape} (features)")
print(f"  y_val: {y_val.shape} (target)")
print(f"  Churn=1: {y_val.sum()} ({y_val.mean()*100:.2f}%)")
print(f"  Churn=0: {len(y_val) - y_val.sum()} ({(1-y_val.mean())*100:.2f}%)")

print(f"\nTest data (for final predictions):")
print(f"  X_test: {X_test_processed.shape}")

print("\n✅ Train/Test split complete!")

In [ ]:
# Random Forest - SEQUENTIAL - TRAINING:



# PART 1: BASELINE (SEQUENTIAL - NO PARALLELIZATION)


import time


print("\n" + "=" * 70)
print("PART 1: BASELINE (SEQUENTIAL)")
print("=" * 70)

print("\n--- 1.1: Random Forest (Sequential, n_jobs=1) ---")
start_time = time.time()

rf_sequential = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    class_weight='balanced',
    random_state=42,
    n_jobs=1  # ← SEQUENTIAL (uses only 1 core)
)

rf_sequential.fit(X_train, y_train)
sequential_rf_time = time.time() - start_time

y_pred_seq = rf_sequential.predict_proba(X_val)[:, 1]
roc_auc_seq = roc_auc_score(y_val, y_pred_seq)

print(f"Training Time: {sequential_rf_time:.2f} seconds")
print(f"ROC-AUC Score: {roc_auc_seq:.4f}")


In [ ]:
#SEQUENTIAL LOGESTIC REGRESSION - TRAINING:

print("\n--- 1.2: Logistic Regression (Sequential) ---")
start_time = time.time()

lr_sequential = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',
    random_state=42,
    n_jobs=1  # ← SEQUENTIAL
)

lr_sequential.fit(X_train, y_train)
sequential_lr_time = time.time() - start_time

y_pred_lr_seq = lr_sequential.predict_proba(X_val)[:, 1]
roc_auc_lr_seq = roc_auc_score(y_val, y_pred_lr_seq)

print(f"Training Time: {sequential_lr_time:.2f} seconds")
print(f"ROC-AUC Score: {roc_auc_lr_seq:.4f}")


In [ ]:
# PART 2: PARALLELIZED (n_jobs=-1)
